In [1]:
!pip install -q "transformers==5.0.0" datasets accelerate kagglehub

In [2]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed,
)

set_seed(42)
random.seed(42)
np.random.seed(42)

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 256
N_SAMPLE = 1500

TARGET_COLS = [
    "structure_score",
    "relevance_score",
    "overall"
]

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)
print("Torch:", torch.__version__)

Device: cuda
Torch: 2.10.0+cu128


In [3]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "aryan208/hr-interview-questions-and-ideal-answers"
)

print("Dataset:", dataset_path)

data_file = os.path.join(
    dataset_path,
    "hr_interview_questions_dataset.json"
)

print("File exists:", os.path.exists(data_file))

Mounting files to /kaggle/input/datasets/aryan208/hr-interview-questions-and-ideal-answers...
Dataset: /kaggle/input/datasets/aryan208/hr-interview-questions-and-ideal-answers
File exists: True


In [4]:
with open(data_file, "r") as f:
    data = json.load(f)

raw_df = pd.DataFrame(data)

print("Shape:", raw_df.shape)
print(raw_df.columns.tolist())

raw_df.head()

Shape: (2500000, 8)
['question', 'category', 'role', 'experience', 'difficulty', 'source_type', 'ideal_answer', 'keywords']


,question,category,role,experience,difficulty,source_type,ideal_answer,keywords
0,Tell me about a time you had to learn somethin...,Adaptability,DevOps Engineer,fresher,Easy,Open-Ended,I'm always eager to learn and embrace change a...,"[flexible, change]"
1,Describe a time you handled a difficult situat...,Conflict Resolution,Product Manager,2 years,Hard,Behavioral,"When faced with conflict, I approach it calmly...","[problem, disagree]"
2,Where do you see yourself in 5 years?,Career Goals,Data Scientist,2 years,Medium,Open-Ended,My long-term goal is to grow into a senior rol...,"[aspiration, dream]"
3,Tell me about a conflict you had with a cowork...,Conflict Resolution,Marketing Associate,4 years,Hard,STAR-Based,"When faced with conflict, I approach it calmly...","[problem, disagree]"
4,What skills do you hope to develop in your nex...,Career Goals,Marketing Associate,1 year,Easy,Situational,My long-term goal is to grow into a senior rol...,"[future, aspiration]"


In [5]:
raw_df = raw_df.rename(columns={
    "question": "question",
    "ideal_answer": "ideal_answer"
})

raw_df = raw_df[
    raw_df["question"].notna() &
    raw_df["ideal_answer"].notna()
].copy()

print("Valid rows:", len(raw_df))


Valid rows: 2500000


In [6]:
GENERIC_FILLERS = [
    "I am a hardworking person and I work well with everyone. "
    "I always try my best and I believe I would be a good fit.",

    "I always stay positive and do my work properly. "
    "I believe teamwork and communication are very important.",

    "I try to learn new things and complete my responsibilities. "
    "I always give my best in every situation."
]


def jitter(value, spread=0.05):
    return float(
        np.clip(
            value + np.random.uniform(-spread, spread),
            0,
            1
        )
    )


def truncate_answer(text):
    sentences = [
        s.strip()
        for s in text.split(".")
        if s.strip()
    ]

    if sentences:
        return sentences[0] + "."

    return text


def shuffle_answer(text):
    sentences = [
        s.strip()
        for s in text.split(".")
        if s.strip()
    ]

    if len(sentences) <= 1:
        return text

    random.shuffle(sentences)

    return ". ".join(sentences) + "."


def make_input(question, answer):
    return f"Question: {question}\nAnswer: {answer}"


def build_dataset(df, n=1500):

    df = df.sample(
        min(n, len(df)),
        random_state=42
    ).reset_index(drop=True)

    rows = []

    for i, row in df.iterrows():

        question = str(row["question"])
        answer = str(row["ideal_answer"])

        if len(answer.split()) < 6:
            continue

        # 1. Good answer
        rows.append({
            "text": make_input(question, answer),
            "structure_score": jitter(0.92),
            "relevance_score": jitter(0.92),
            "overall": jitter(0.92)
        })

        # 2. Truncated answer
        bad_answer = truncate_answer(answer)

        rows.append({
            "text": make_input(question, bad_answer),
            "structure_score": jitter(0.30),
            "relevance_score": jitter(0.75),
            "overall": jitter(0.45)
        })

        # 3. Shuffled answer
        bad_answer = shuffle_answer(answer)

        rows.append({
            "text": make_input(question, bad_answer),
            "structure_score": jitter(0.35),
            "relevance_score": jitter(0.70),
            "overall": jitter(0.45)
        })

        # 4. Wrong answer
        other_idx = random.randrange(len(df))

        if other_idx != i:

            other_answer = str(
                df.iloc[other_idx]["ideal_answer"]
            )

            rows.append({
                "text": make_input(question, other_answer),
                "structure_score": jitter(0.70),
                "relevance_score": jitter(0.15),
                "overall": jitter(0.30)
            })

        # 5. Generic answer
        filler = random.choice(GENERIC_FILLERS)

        rows.append({
            "text": make_input(question, filler),
            "structure_score": jitter(0.15),
            "relevance_score": jitter(0.20),
            "overall": jitter(0.15)
        })

    return pd.DataFrame(rows)


df = build_dataset(raw_df, N_SAMPLE)

print("Generated examples:", len(df))

df.head()

Generated examples: 7500


,text,structure_score,relevance_score,overall
0,Question: What drives your passion for this fi...,0.907454,0.965071,0.943199
1,Question: What drives your passion for this fi...,0.309866,0.715602,0.415599
2,Question: What drives your passion for this fi...,0.305808,0.736618,0.460112
3,Question: What drives your passion for this fi...,0.720807,0.102058,0.346991
4,Question: What drives your passion for this fi...,0.183244,0.171234,0.118182


In [7]:
train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(val_df))

Train: 6375
Validation: 1125


In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )


train_encodings = tokenizer(
    train_df["text"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=MAX_LEN
)

val_encodings = tokenizer(
    val_df["text"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=MAX_LEN
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [10]:
class SimpleDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.astype(np.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(value[idx])
            for key, value in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx],
            dtype=torch.float32
        )

        return item


train_dataset = SimpleDataset(
    train_encodings,
    train_df[TARGET_COLS].values
)

val_dataset = SimpleDataset(
    val_encodings,
    val_df[TARGET_COLS].values
)

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    problem_type="regression"
)

model.to(device)

print("Model loaded!")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded!


In [12]:
def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    predictions = np.asarray(predictions)
    labels = np.asarray(labels)

    mae = np.mean(
        np.abs(predictions - labels),
        axis=0
    )

    return {
        "structure_mae": float(mae[0]),
        "relevance_mae": float(mae[1]),
        "overall_mae": float(mae[2]),
        "mean_mae": float(mae.mean())
    }

In [13]:
training_args = TrainingArguments(
    output_dir="./star_quality_model",

    num_train_epochs=3,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    learning_rate=2e-5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="mean_mae",
    greater_is_better=False,

    logging_steps=50,

    report_to="none",

    fp16=torch.cuda.is_available()
)


trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    compute_metrics=compute_metrics
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Structure Mae,Relevance Mae,Overall Mae,Mean Mae
1,0.017179,0.010940,0.039782,0.048115,0.044944,0.044280
2,0.014977,0.010586,0.038788,0.036243,0.039418,0.038150
3,0.013448,0.010528,0.038532,0.037749,0.039242,0.038508


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=600, training_loss=0.02621078133583069, metrics={'train_runtime': 252.2621, 'train_samples_per_second': 75.814, 'train_steps_per_second': 2.378, 'total_flos': 1266742089792000.0, 'train_loss': 0.02621078133583069, 'epoch': 3.0})

In [14]:
results = trainer.evaluate()

for key, value in results.items():

    if isinstance(value, float):
        print(f"{key}: {value:.4f}")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


eval_loss: 0.0106
eval_structure_mae: 0.0388
eval_relevance_mae: 0.0362
eval_overall_mae: 0.0394
eval_mean_mae: 0.0381
eval_runtime: 4.6246
eval_samples_per_second: 243.2670
eval_steps_per_second: 3.8920
epoch: 3.0000


In [15]:
def predict_quality(question, answer):

    text = make_input(question, answer)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LEN
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    model.eval()

    with torch.no_grad():

        outputs = model(**inputs)

        scores = outputs.logits.squeeze().cpu().numpy()

    scores = np.clip(scores, 0, 1)

    return {
        "structure_score": float(scores[0]),
        "relevance_score": float(scores[1]),
        "overall": float(scores[2])
    }

In [16]:
question = "Tell me about a time you solved a difficult problem."

answer = """
Our checkout page had a 12% cart-abandonment spike.
I was asked to identify the cause within two weeks.
I analyzed the funnel and found a broken coupon field on mobile.
I worked with the payments team to fix it.
Cart abandonment returned to baseline within three days.
"""

scores = predict_quality(
    question,
    answer
)

scores


{'structure_score': 0.3889836072921753,
 'relevance_score': 0.5564097762107849,
 'overall': 0.39370816946029663}

In [17]:
SAVE_DIR = "./star_quality_final"

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Model saved to:", SAVE_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: ./star_quality_final
